# Training Environment Setup

In [1]:
import os
import sys
import json
import math
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import DataLoader
from pathlib import Path

In [2]:
PROJECT_ROOT = Path.cwd().parent

sys.path.insert(0,str(PROJECT_ROOT))
from src import (
    CelebADataset,
    get_train_transforms,
    get_eval_transforms,
    FaceRecognitionModel
)

DATASET_DIR = PROJECT_ROOT / "data" / "celeba"
PROCESSED_DIR = DATASET_DIR / "processed"
CHECKPOINT_DIR = PROJECT_ROOT / "checkpoints"

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

print("Project:", PROJECT_ROOT)
print("Dataset:", PROCESSED_DIR)
print("Checkpoints:", CHECKPOINT_DIR)

Project: Y:\Python\AI ML\Month 2\week 8-NLP\45 — APPLE FACE ID  (FACIAL RECOGNITION PROJECTS)
Dataset: Y:\Python\AI ML\Month 2\week 8-NLP\45 — APPLE FACE ID  (FACIAL RECOGNITION PROJECTS)\data\celeba\processed
Checkpoints: Y:\Python\AI ML\Month 2\week 8-NLP\45 — APPLE FACE ID  (FACIAL RECOGNITION PROJECTS)\checkpoints


In [3]:
train_df = pd.read_csv(PROCESSED_DIR / "train.csv")
validation_df = pd.read_csv(PROCESSED_DIR / "validation.csv")
test_df = pd.read_csv(PROCESSED_DIR / "test.csv")

with open(PROCESSED_DIR / "identity_to_class.json") as file:
    identity_to_class = {
        int(k): int(v) for k, v in json.load(file).items()
    }

with open(PROCESSED_DIR / "dataset_config.json") as file:
    dataset_config = json.load(file)

NUM_CLASSES = len(identity_to_class)
IMAGE_SIZE = dataset_config["image_size"]
NUM_CLASSES = dataset_config["num_classes"]

In [4]:
EMBEDDING_SIZE = 128
NUM_CLASSES = len(identity_to_class)

print("Training images:", len(train_df))
print("Training identities:", NUM_CLASSES)
print("Validation images:", len(validation_df))
print("Test images:", len(test_df))

Training images: 131442
Training identities: 5103
Validation images: 19867
Test images: 19962


In [5]:
BATCH_SIZE = 32
NUM_WORKERS = 0

train_dataset = CelebADataset(
    dataframe=train_df,
    transform=get_train_transforms(IMAGE_SIZE),
    label_mapping=identity_to_class
)

validation_dataset = CelebADataset(
    dataframe=validation_df,
    transform=get_eval_transforms(IMAGE_SIZE)
)

test_dataset = CelebADataset(
    dataframe=test_df,
    transform=get_eval_transforms(IMAGE_SIZE)
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    drop_last=True
)

validation_loader = DataLoader(
    validation_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)

In [6]:
DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = FaceRecognitionModel(
    embedding_size=EMBEDDING_SIZE,
    num_classes=NUM_CLASSES
).to(DEVICE)

print("Device:", DEVICE)
print("Training batches:", len(train_loader))

Device: cuda
Training batches: 4107


# Loss Function, Optimizer & Learning Rate Scheduler

In [7]:
EPOCHS = 20

LEARNING_RATE = 0.01
WEIGHT_DECAY = 0.0005

ARC_MARGIN = 0.5
ARC_SCALE = 64.0

In [8]:
criterion = nn.CrossEntropyLoss()

In [9]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

In [10]:
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=EPOCHS,
    eta_min=1e-6
)

# Training Loop Implementation

In [11]:
from tqdm.auto import tqdm

def train_one_epoch(model, train_loader, criterion, optimizer, device):

    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    progress_bar = tqdm(train_loader, desc="Training")

    for images, labels in progress_bar:

        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        # Reset gradients
        optimizer.zero_grad(set_to_none=True)

        # Forward pass
        logits, embeddings = model(images, labels)

        # Calculate loss
        loss = criterion(logits, labels)

        # Backpropagation
        loss.backward()

        # Update model parameters
        optimizer.step()

        # Calculate statistics
        batch_size = labels.size(0)

        running_loss += loss.item() * batch_size

        predictions = torch.argmax(logits, dim=1)

        correct += (predictions == labels).sum().item()
        total += batch_size

        progress_bar.set_postfix(
            loss=f"{loss.item():.4f}",
            accuracy=f"{100 * correct / total:.2f}%"
        )

    epoch_loss = running_loss / total
    epoch_accuracy = 100 * correct / total

    return epoch_loss, epoch_accuracy

# Epoch Training and Validation

In [14]:
# Generate validation pairs

# Reset validation dataframe index
validation_df = validation_df.reset_index(drop=True)

# Group image indices by identity
identity_groups = {
    identity: group.index.to_numpy()
    for identity, group in validation_df.groupby("identity_id")
}

# Only identities having at least 2 images can form genuine pairs
eligible_identities = [
    identity
    for identity, indices in identity_groups.items()
    if len(indices) >= 2
]

rng = np.random.default_rng(42)

# Generate genuine pairs
genuine_pairs = []

for identity in eligible_identities:

    indices = identity_groups[identity]

    for _ in range(5):
        image1, image2 = rng.choice(indices, size=2, replace=False)

        genuine_pairs.append([int(image1), int(image2)])

# Generate an equal number of impostor pairs
impostor_pairs = []

for _ in range(len(genuine_pairs)):

    identity1, identity2 = rng.choice(
        eligible_identities,
        size=2,
        replace=False
    )

    image1 = rng.choice(identity_groups[identity1])
    image2 = rng.choice(identity_groups[identity2])

    impostor_pairs.append([int(image1), int(image2)])

# Combine pairs and assign labels
validation_pairs = np.array(
    genuine_pairs + impostor_pairs,
    dtype=np.int64
)

pair_labels = np.array(
    [1] * len(genuine_pairs) +
    [0] * len(impostor_pairs),
    dtype=np.int64
)

# Shuffle pairs
shuffle_indices = rng.permutation(len(validation_pairs))

validation_pairs = validation_pairs[shuffle_indices]
pair_labels = pair_labels[shuffle_indices]

print("Genuine pairs:", len(genuine_pairs))
print("Impostor pairs:", len(impostor_pairs))
print("Total validation pairs:", len(validation_pairs))

Genuine pairs: 4900
Impostor pairs: 4900
Total validation pairs: 9800


In [18]:
from sklearn.metrics import roc_auc_score

def validate_one_epoch(model, validation_loader, validation_pairs,
                       pair_labels, device):

    # Extract embeddings for all validation images
    embeddings, identities = extract_embeddings(
        model=model,
        data_loader=validation_loader,
        device=device
    )

    # Convert pair indices into tensors
    pair_indices = torch.tensor(
        validation_pairs,
        dtype=torch.long
    )

    # Retrieve embeddings for both images
    embeddings_1 = embeddings[pair_indices[:, 0]]
    embeddings_2 = embeddings[pair_indices[:, 1]]

    # Calculate cosine similarity
    similarity_scores = F.cosine_similarity(
        embeddings_1,
        embeddings_2,
        dim=1
    )

    # Calculate ROC-AUC
    validation_auc = roc_auc_score(
        pair_labels,
        similarity_scores.numpy()
    )

    # Calculate average similarity scores
    genuine_scores = similarity_scores[pair_labels == 1]
    impostor_scores = similarity_scores[pair_labels == 0]

    return (
        validation_auc,
        genuine_scores.mean().item(),
        impostor_scores.mean().item()
    )

In [36]:
def extract_embeddings(model, data_loader, device):

    model.eval()

    all_embeddings = []
    all_identities = []

    with torch.inference_mode():

        for images, identities in tqdm(data_loader, desc="Extracting embeddings"):

            images = images.to(device)

            embeddings = model.backbone(images)

            embeddings = F.normalize(embeddings, p=2, dim=1)

            all_embeddings.append(embeddings.cpu())
            all_identities.extend(identities.numpy())

    all_embeddings = torch.cat(all_embeddings, dim=0)

    return all_embeddings, np.array(all_identities)

In [37]:
history = {
    "train_loss": [],
    "train_accuracy": [],
    "validation_auc": [],
    "genuine_similarity": [],
    "impostor_similarity": [],
    "learning_rate": []
}

In [38]:
def train_model(
    model,
    train_loader,
    validation_loader,
    validation_pairs,
    pair_labels,
    criterion,
    optimizer,
    scheduler,
    history,
    completed_epoch,
    global_step,
    device,
    total_epochs
):

    if completed_epoch >= total_epochs:

        print("=" * 50)
        print("TRAINING ALREADY COMPLETED")
        print("=" * 50)

        print(f"Completed Epochs : {completed_epoch}/{total_epochs}")
        print(f"Global Steps     : {global_step}")
        print(f"Learning Rate    : {optimizer.param_groups[0]['lr']:.8f}")

        print("\nNo additional training required.")

        return completed_epoch, global_step

    for epoch in range(completed_epoch, total_epochs):

        train_loss, train_accuracy = train_one_epoch(
            model,
            train_loader,
            criterion,
            optimizer,
            device
        )

        global_step += len(train_loader)

        validation_auc, genuine_similarity, impostor_similarity = validate_one_epoch(
            model,
            validation_loader,
            validation_pairs,
            pair_labels,
            device
        )

        scheduler.step()

        history["train_loss"].append(train_loss)
        history["train_accuracy"].append(train_accuracy)
        history["validation_auc"].append(validation_auc)
        history["genuine_similarity"].append(genuine_similarity)
        history["impostor_similarity"].append(impostor_similarity)
        history["learning_rate"].append(optimizer.param_groups[0]["lr"])

        completed_epoch = epoch + 1

        print(f"\nEpoch [{completed_epoch}/{total_epochs}]")
        print(f"Training Loss      : {train_loss:.4f}")
        print(f"Training Accuracy  : {train_accuracy:.2f}%")
        print(f"Validation AUC     : {validation_auc:.4f}")
        print(f"Genuine Similarity : {genuine_similarity:.4f}")
        print(f"Impostor Similarity: {impostor_similarity:.4f}")
        print(f"Learning Rate      : {optimizer.param_groups[0]['lr']:.8f}")

    return completed_epoch, global_step

In [39]:
checkpoint_path = CHECKPOINT_DIR / "epoch_checkpoint.pth"

checkpoint = torch.load(
    checkpoint_path,
    map_location=DEVICE,
    weights_only=False
)

model.load_state_dict(checkpoint["model_state_dict"])

optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

scheduler.load_state_dict(checkpoint["scheduler_state_dict"])

completed_epoch = checkpoint["completed_epoch"]
global_step = checkpoint["global_step"]

assert checkpoint["identity_to_class"] == identity_to_class

print("Training checkpoint loaded successfully!")
print(f"Completed Epochs : {completed_epoch}/{EPOCHS}")
print(f"Global Steps     : {global_step}")
print(f"Learning Rate    : {optimizer.param_groups[0]['lr']:.8f}")

Training checkpoint loaded successfully!
Completed Epochs : 20/20
Global Steps     : 54780
Learning Rate    : 0.00000100


In [40]:
completed_epoch, global_step = train_model(
    model=model,
    train_loader=train_loader,
    validation_loader=validation_loader,
    validation_pairs=validation_pairs,
    pair_labels=pair_labels,
    criterion=criterion,
    optimizer=optimizer,
    scheduler=scheduler,
    history=history,
    completed_epoch=completed_epoch,
    global_step=global_step,
    device=DEVICE,
    total_epochs=EPOCHS
)

TRAINING ALREADY COMPLETED
Completed Epochs : 20/20
Global Steps     : 54780
Learning Rate    : 0.00000100

No additional training required.


In [41]:
latest_checkpoint_path = CHECKPOINT_DIR / "latest_checkpoint.pth"

checkpoint_to_save = {
    "completed_epoch": completed_epoch,
    "global_step": global_step,
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "scheduler_state_dict": scheduler.state_dict(),
    "identity_to_class": identity_to_class
}

torch.save(checkpoint_to_save, latest_checkpoint_path)

print("Latest checkpoint saved successfully!")
print("Path:", latest_checkpoint_path)
print("Completed Epochs:", completed_epoch)
print("Global Steps:", global_step)

Latest checkpoint saved successfully!
Path: Y:\Python\AI ML\Month 2\week 8-NLP\45 — APPLE FACE ID  (FACIAL RECOGNITION PROJECTS)\checkpoints\latest_checkpoint.pth
Completed Epochs: 20
Global Steps: 54780
